# பாடம் 08 - பன்முக பிரதிநிதி வடிவமைப்பு முறை


## அமைப்பு


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## பல முகவர் அமைப்புகள் ஏன்?

 பயண திட்டமிடல் போன்ற உண்மைக் காரியங்களில் பல்வேறு வகையான நிபுணத்துவங்கள் தேவை - பொருளாதாரம், உள்ளூர் அறிவு, பட்ஜெட் மற்றும் பல. அனைத்தையும் ஒரே முகவர் கையாள முயன்றால் அது விரைவில் கையாளக்கூடியதாக болмайி சிக்கலாகிவிடும்.

பல முகவர் அமைப்புகள் இதைக் **திறம்படத்தன்மை** மூலம் தீர்க்கின்றன: ஒவ்வொரு முகவரும் தனக்கு உகந்த நிபுணத்துவ பகுதியில் கவனம் செலுத்தி, பொதுவானவரைவிட உயர் தரமான முடிவுகளை தருகின்றன. மேலும் அவை **எளிதில் விரிவடையக்கூடிய**வையாகும் — புதிய முகவர்களை (எ.கா., விமான நிபுணர், உணவக விமர்சகர்) தற்போதைய பணிப்பகுச்செயல்முறையை மாற்றிக்கொள்ளாமல் சேர்க்க முடியும். இவையெல்லாம் குறுந்தகடு வழியாக ஒன்றோடொன்று இணைந்து செயல்பட்டு, பண்பட்ட தகவலை ஒருவர் முதல் அடுத்தவருக்கு ஒப்படைக்கும்.


## சிறப்பு முகவர்களை உருவாக்குதல்


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## தொடர் பணிவழி ஒன்றை கட்டமைத்தல்

`WorkflowBuilder` உங்கள் முகவரிகளை ஒரு திசையான வரைபடத்தில் இணைக்க அனுமதிக்கிறது. இங்கே நாம் ஒரு எளிய இரு படி குழாய்தசையை உருவாக்குகிறோம்: **TravelPlanner** பயண திட்டத்தை வரைந்து, அதன் பிறகு **TravelConcierge** அதை பரிசீலித்து மேம்படுத்துகிறது.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## வேலைநடவடிக்கையில் மேலும் முகவர்களைச் சேர்த்தல்

பல முகவர்கள் முறைமை முன்னிலை பெற்ற ஒரு முக்கியமான நன்மை அதன் விரிவாக்கத் திறன் ஆகும். கீழே, பயணியின் பட்ஜெட்டிற்குப் பொருந்துவதைச் சரிபார்ப்பதற்கும், செலவுக்கு எல்லையை மீறக்கூடிய உருப்படிகளை குறிக்கவும், பணத்தைச் சேமிக்க கூடிய மாற்றுகளை பரிந்துரைக்கவும் **BudgetReviewer** முகவரைக் கூருகின்றோம். வேலைநடவடிக்கை இப்போது மூன்று முகவர்களை தொடர்ச்சியாக இயக்குகிறது:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## சுருக்கம்

இந்த பாடத்தில் நீங்கள் எவ்வாறு கற்றுக்கொண்டீர்கள்:

1. **தனிச்செயல்திறனுடைய முகவர்களை உருவாக்குதல்** — ஒவ்வொருவரும் ஒரு குறிப்பிட்ட பங்கு (திட்டமிடல், காவலாளர், பட்ஜெட்டின் மதிப்பாய்வு) கொண்டவர்கள்.
2. **`WorkflowBuilder` மற்றும் `add_edge` பயன்படுத்தி முகவர்களை தொடர்ச்சியான வேலைநிறுவலை ஆக்கியல் இணைக்குதல்**.
3. **பல முகவர்களின் குழாயிலிருந்து வெளியீட்டை நேரடியாக வழங்குதல்**, பேசும் முகவரைக் கண்காணித்தல்.
4. **வேலைநிறுவலை விரிவாக்குதல்** — புதிய முகவர்களை தொடரில் சேர்த்து முந்தைய முகவர்களை மாற்றாமல் வைப்பது.

பல முகவர் வடிவமைப்பு மாதிரி ஒவ்வொரு முகவரையும் எளிதாக்கும் போது, தனித்து செயல்படும் எந்த ஒரு முகவரும் எட்ட முடியாத வளமான, விரிவாக மதிப்பாய்வுசெய்யப்பட்ட முடிவுகளை உருவாக்குகிறது.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**எச்சரிக்கை**:  
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவையான [Co-op Translator](https://github.com/Azure/co-op-translator) மூலம் மொழிபெயர்க்கப்பட்டது. நம்மால் சரியான மொழிபெயர்ப்பு தர முயலினாலும், தானாக நிகழும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை தயவுசெய்து கருத்தில் கொள்ளவும். அச்சாணிக்கும் தாய்மொழியில் உள்ள அடிப்படையான ஆவணம் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்முறை மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதனால் ஏற்படும் எந்தவொரு தவறான புரிதலுக்கோ அல்லது தவறான பொருள் எடுத்துக்கொள்ளலுக்கோ நாங்கள் பொறுப்பற்றவர்கள்.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
